In [1]:
import pandas as pd
import numpy as np
from thermoengine import model, equilibrate, core
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import timeout_decorator


In [2]:
modelDB = model.Database(liq_mod='pMELTS')

In [3]:
Liq = modelDB.get_phase('Liq')
Ol = modelDB.get_phase('Ol')
Cpx = modelDB.get_phase('Cpx')
Opx = modelDB.get_phase('Opx')
Fsp = modelDB.get_phase('Fsp')
Spn = modelDB.get_phase('SplS')
Grt = modelDB.get_phase('Grt')
# Fo = modelDB.get_phase('Fo')

In [4]:
elm_sys = ['H','O','Na','Mg','Al','Si','P','K','Ca','Ti','Cr','Mn','Fe','Co','Ni']
phs_sys = [Liq, Ol, Cpx, Opx, Fsp, Spn, Grt]
phs_sys_simple = [Liq, Cpx, Opx, Fsp, Spn, Grt, Ol]
phase_objs = {'liq': Liq, 'ol': Ol, 'cpx': Cpx, 'opx': Opx, 'pl': Fsp, 'spn': Spn, 'g': Grt}
phase_objs_simple = {'liq': Liq, 'ol': Ol, 'cpx': Cpx, 'opx': Opx, 'pl': Fsp, 'spn': Spn, 'g': Grt}

In [5]:
# KLB-1 Davis et al. (2009)
lz_grm_oxides = {
    'SiO2':  44.84, 
    'TiO2':   0.11, 
    'Al2O3':  3.51, 
    'Fe2O3':  8.20*0.032/(55.845+16)*(55.845*2+16*3)/2,
    'Cr2O3':  0.32, 
    'FeO':    8.20*(1-0.032), 
    'MnO':    0.12,
    'MgO':   39.52, 
    'NiO':    0.0, 
    'CoO':    0.0,
    'CaO':    3.07, 
    'Na2O':   0.3, 
    'K2O':    0.02, 
    'P2O5':   0.0, 
    'H2O':    0.0
}

# lz_grm_oxides ={'SiO2':  44.84, 
#        'TiO2':   0.11, 
#        'Al2O3':  3.51, 
#        'Fe2O3':  8.20*0.032/(55.845+16)*(55.845*2+16*3)/2,
#        # 'Cr2O3':  0.32, 
#         'Cr2O3':  0.0, 
#        'FeO':    8.20*(1-0.032), 
#        # 'MnO':    0.12,
#         'MnO': 0.0,
#        'MgO':   39.52, 
#        'NiO':    0.0, 
#        'CoO':    0.0,
#        'CaO':    3.07, 
#        'Na2O':   0.3, 
#        # 'K2O':    0.02, 
#         'K2O': 0.0,
#        'P2O5':   0.0, 
#        # 'H2O':    0.0
#        }

In [6]:
lz_mol_oxides = core.chem.format_mol_oxide_comp(lz_grm_oxides, convert_grams_to_moles=True)
lz_moles_end, lz_oxide_res = Liq.calc_endmember_comp(
    mol_oxide_comp=lz_mol_oxides, method='intrinsic', output_residual=True)
if not Liq.test_endmember_comp(lz_moles_end):
    print ("Calculated composition is infeasible!")
lz_mol_elm = Liq.convert_endmember_comp(lz_moles_end,output='moles_elements')

In [7]:
lz_blk_cmp = []
for elm in elm_sys:
    index = core.chem.PERIODIC_ORDER.tolist().index(elm)
    lz_blk_cmp.append(lz_mol_elm[index])
lz_blk_cmp = np.array(lz_blk_cmp)



In [8]:
phase_names = {'liq': 'Liquid', 'ol': 'Olivine', 'cpx': 'Clinopyroxene', 
               'opx': 'Orthopyroxene', 'pl': 'Feldspar', 'spn': 'Spinel', 'g': 'Garnet'}

In [9]:
@timeout_decorator.timeout(5)
def run_calc(t, p, bulk_comp):
    equil = equilibrate.Equilibrate(elm_sys, phs_sys)
    state = equil.execute(t, p, bulk_comp=bulk_comp, debug=0, stats=False)
    return state

In [10]:
p = np.arange(0.0, 80000.0, 500.0)
p[0] = 1.0
t = np.arange(1080.0, 2500.0, 10.0) + 273.15
t = t[::-1]

print(np.shape(p))
print(np.shape(t))

(160,)
(142,)


In [11]:
# results = pd.DataFrame()
results = pd.read_csv('results_backup.csv')

prev_f = 1.0

for k in range(np.shape(p)[0]):
    print(str(k+1) + ' of ' + str(np.shape(p)[0]))
    if k > 86:

        for j in range(np.shape(t)[0]):
            if prev_f > 0.005 or j == 0:
                try:
                    state = run_calc(t[j], p[k], lz_blk_cmp)

                    row = {}
                    row['pressure'] = state.pressure / 1e4
                    row['temperature'] = state.temperature - 273.15

                    # Phase mass proportions
                    total_mass = 0.0
                    for ph in phase_names.keys():
                        row[ph+'_mass'] = state.tot_grams_phase(phase_names[ph])
                        total_mass += state.tot_grams_phase(phase_names[ph])

                    for ph in phase_names.keys():
                        row[ph+'_mass'] = row[ph+'_mass'] / total_mass

                    # Phase compositions
                    for ph in phase_names.keys():
                        if ph == 'liq':
                            prefix = 'endm'
                        else:
                            prefix = ''
                        mol_endmembers = state.compositions(phase_names[ph], 'components')
                        component_names = phase_objs[ph].endmember_names
                        for i in range(len(mol_endmembers)):
                            row[ph + '_' + prefix + component_names[i]] = mol_endmembers[i]

                        oxide_comp = state.compositions(phase_names[ph], 'oxides', 'wt%')
                        for i in range(len(core.chem.OXIDE_ORDER)):
                            if core.chem.OXIDE_ORDER[i] in lz_grm_oxides.keys() and lz_grm_oxides[core.chem.OXIDE_ORDER[i]] > 0:
                                row[ph + '_' + core.chem.OXIDE_ORDER[i] + '_wtpt'] = oxide_comp[i]

                    results = results.append(row, ignore_index=True)
                    prev_f = row['liq_mass']
                except Exception:
                    continue

                results.to_csv('results_backup.csv')

1 of 160
2 of 160
3 of 160
4 of 160
5 of 160
6 of 160
7 of 160
8 of 160
9 of 160
10 of 160
11 of 160
12 of 160
13 of 160
14 of 160
15 of 160
16 of 160
17 of 160
18 of 160
19 of 160
20 of 160
21 of 160
22 of 160
23 of 160
24 of 160
25 of 160
26 of 160
27 of 160
28 of 160
29 of 160
30 of 160
31 of 160
32 of 160
33 of 160
34 of 160
35 of 160
36 of 160
37 of 160
38 of 160
39 of 160
40 of 160
41 of 160
42 of 160
43 of 160
44 of 160
45 of 160
46 of 160
47 of 160
48 of 160
49 of 160
50 of 160
51 of 160
52 of 160
53 of 160
54 of 160
55 of 160
56 of 160
57 of 160
58 of 160
59 of 160
60 of 160
61 of 160
62 of 160
63 of 160
64 of 160
65 of 160
66 of 160
67 of 160
68 of 160
69 of 160
70 of 160
71 of 160
72 of 160
73 of 160
74 of 160
75 of 160
76 of 160
77 of 160
78 of 160
79 of 160
80 of 160
81 of 160
82 of 160
83 of 160
84 of 160
85 of 160
86 of 160
87 of 160
88 of 160
89 of 160
90 of 160
91 of 160
92 of 160
93 of 160
94 of 160
95 of 160
96 of 160
97 of 160
98 of 160
99 of 160
100 of 160
101 of 1

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1066fe280>>
Traceback (most recent call last):
  File "/Users/sm905/opt/anaconda3/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
  File "/Users/sm905/opt/anaconda3/lib/python3.9/site-packages/timeout_decorator/timeout_decorator.py", line 69, in handler
    _raise_exception(timeout_exception, exception_message)
  File "/Users/sm905/opt/anaconda3/lib/python3.9/site-packages/timeout_decorator/timeout_decorator.py", line 45, in _raise_exception
    raise exception()
timeout_decorator.timeout_decorator.TimeoutError: 'Timed Out'


KeyboardInterrupt: 